# Model Evaluation — Retail Demand Forecasting

Detailed evaluation of the champion registered model: residual analysis,
feature importance, prediction distribution.

**Reads from:** `gold_ml_features`, `gold_ml_scoring_meta`, MLflow registry

**Writes to:** `gold_ml_feature_importance`

In [ ]:
import json
import mlflow
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
# Load champion model from the MLflow model registry (latest version)
meta = json.loads(spark.read.table('gold_ml_scoring_meta').collect()[0]['meta_json'])
champion_name = meta['champion_model']
feature_cols = meta['feature_cols']
numeric_features = meta['numeric_features']
cat_cols = meta['cat_cols']
category_maps = meta['category_maps']

model = mlflow.pyfunc.load_model(f'models:/{champion_name}/latest')
print(f'Loaded registered model: {champion_name} (latest version)')

In [ ]:
# Rebuild the same pandas feature frame as training (named float64 columns)
LABEL = 'daily_quantity'
pdf = spark.read.table('gold_ml_features').select(numeric_features + cat_cols + [LABEL]).toPandas()
for c in numeric_features:
    pdf[c] = pd.to_numeric(pdf[c], errors='coerce').fillna(0.0)
for c in cat_cols:
    pdf[f'{c}_idx'] = pdf[c].astype(str).map(category_maps[c]).fillna(-1).astype('float64')

X = pdf[feature_cols].astype('float64')
y = pdf[LABEL].astype('float64')

# Same split + seed as training, so this is the same held-out validation set
_, X_val, _, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
preds = pd.DataFrame({'daily_quantity': y_val.values, 'prediction': model.predict(X_val)})
print(f'Test predictions: {len(preds):,} rows')

In [ ]:
# Residual analysis
preds['residual'] = preds['daily_quantity'] - preds['prediction']
preds['abs_residual'] = preds['residual'].abs()
nonzero = preds[preds['daily_quantity'] > 0]
preds['pct_error'] = np.where(preds['daily_quantity'] > 0,
                              preds['abs_residual'] / preds['daily_quantity'] * 100, np.nan)

print('=== Residual Statistics ===')
print(preds[['residual', 'abs_residual', 'pct_error']].describe())

mape = (nonzero['abs_residual'] / nonzero['daily_quantity'] * 100).mean()
print(f'MAPE: {mape:.2f}%')

In [ ]:
# Feature importance from the champion model's native flavor.
# pyfunc wraps the raw model; named columns mean importances align 1:1 with feature_cols.
try:
    native = model._model_impl
    booster = getattr(native, 'lgb_model', None) or getattr(native, 'sklearn_model', None) or native
    importances = (booster.feature_importances_
                   if hasattr(booster, 'feature_importances_')
                   else booster.feature_importance(importance_type='gain'))
except Exception:
    importances = np.zeros(len(feature_cols))

imp = np.asarray(importances, dtype='float64')
imp = imp / imp.sum() if imp.sum() > 0 else imp
rows = sorted(zip(feature_cols, [float(v) for v in imp]), key=lambda r: r[1], reverse=True)

print('=== Top 10 Features by Importance ===')
for name, v in rows[:10]:
    print(f'  {name:30s} {v:.4f}')

fi_spark = (
    spark.createDataFrame(rows, ['feature', 'importance'])
    .withColumn('model_timestamp', current_timestamp())
)
fi_spark.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_feature_importance')
print('Feature importance saved to gold_ml_feature_importance')

In [ ]:
spark.sql('OPTIMIZE gold_ml_feature_importance')
print('Evaluation complete')